# 📈 Notebook 2: Adaptive Failure Detection (phi accrual)

Notebook 1 left us with a dilemma: **a fixed timeout is either too jumpy or too slow**, and the right number changes with the weather (network conditions, GC pauses, deploy storms…).

Hayashibara et al.'s **phi accrual failure detector** (used by Cassandra, Akka, ScyllaDB) takes a different approach:

> Instead of a binary alive/dead flag, output a continuously-rising **suspicion score** `φ`. Pages or actions trigger when `φ` crosses a threshold.

## The intuition

1. Keep a sliding window of recent inter-arrival times (gap between heartbeat 1→2, 2→3, ...).
2. When asked *"is the node alive **right now**?"*, look at how long it's been since the last beat (`t_since_last`). The longer it is, the less plausible it is under the recent distribution.
3. Convert that probability into a score: `φ = -log10(P(interval >= t_since_last))`.

- `φ = 1` ≈ 10% chance the node is just slow.
- `φ = 3` ≈ 1 in 1000 — getting suspicious.
- `φ = 8` ≈ 1 in 100,000,000 — almost certainly dead. (Cassandra's default.)

The threshold becomes a single tunable knob whose meaning is **roughly the same on a fast LAN and a flaky WAN**, because the detector adapts.

## Learning objectives
- Implement a phi accrual detector with a sliding window.
- Compare it side-by-side with the fixed-timeout detector from notebook 1.
- See it shrug off a temporary network blip that would have triggered a fixed-timeout false positive.

In [ ]:
import math, statistics, random
from collections import deque

class PhiDetector:
    """Hayashibara phi accrual failure detector (Normal-distribution version).

    The original paper uses an exponential interval distribution; the
    Cassandra/Akka implementations use a Normal approximation, which is
    what we do here for clarity.
    """

    def __init__(self, window=100):
        self.intervals = deque(maxlen=window)
        self.last_beat = None

    def heartbeat(self, t):
        if self.last_beat is not None:
            self.intervals.append(t - self.last_beat)
        self.last_beat = t

    def phi(self, now):
        # Not enough data yet -> no suspicion
        if self.last_beat is None or len(self.intervals) < 2:
            return 0.0
        mean = statistics.mean(self.intervals)
        std  = max(statistics.pstdev(self.intervals), 1e-3)  # avoid div-by-zero
        delta = now - self.last_beat
        # P(interval >= delta) under Normal(mean, std). The survival function
        # of a Normal is 0.5 * erfc(z / sqrt(2)) where z = (delta - mean)/std.
        z = (delta - mean) / std
        p = 0.5 * math.erfc(z / math.sqrt(2))
        p = max(p, 1e-12)  # cap to avoid log(0)
        return -math.log10(p)

## 1. Same scenario as notebook 1: clean run, node dies at t=30s

We re-use the same heartbeat trace generator: ~1s cadence with 0.4s jitter, dies at `t = 30`. We sample `phi(t)` every 0.1s.

In [ ]:
random.seed(0)
TICK, JITTER, DEAD_AT, TOTAL = 1.0, 0.4, 30.0, 40.0

def make_trace(dead_at=DEAD_AT, total=TOTAL, tick=TICK, jitter=JITTER, blip=None):
    """Generate a list of heartbeat arrival times.

    blip=(start, duration) silences the node for a window in the middle
    of its life - a fake network blip - but does NOT kill it.
    """
    out, t = [], 0.0
    while t < total:
        t += tick + random.uniform(-jitter, jitter)
        if t >= dead_at:
            break
        if blip is not None and blip[0] <= t < blip[0] + blip[1]:
            continue  # heartbeat 'lost' during the blip
        out.append(t)
    return out

heartbeats = make_trace()

detector = PhiDetector()
ts, phis = [], []
hb_iter = iter(heartbeats)
next_hb = next(hb_iter, None)
t = 0.0
while t < TOTAL:
    while next_hb is not None and next_hb <= t:
        detector.heartbeat(next_hb)
        next_hb = next(hb_iter, None)
    ts.append(t)
    phis.append(detector.phi(t))
    t += 0.1

import matplotlib.pyplot as plt
plt.figure(figsize=(9, 4))
plt.plot(ts, phis, label='phi')
plt.axhline(8, color='red', linestyle='--', label='threshold (phi=8)')
plt.axvline(DEAD_AT, color='grey', linestyle=':', label='actually died')
plt.xlabel('time (s)'); plt.ylabel('phi')
plt.title('Phi accrual: suspicion grows smoothly until it crosses the threshold')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

cross = next((tt for tt, v in zip(ts, phis) if v > 8), None)
print(f'phi crossed 8 at t={cross:.1f}s (true death at t={DEAD_AT})')

# Quiet while alive (after warm-up), unbounded after the crash, and no early firing.
assert cross is not None and cross >= DEAD_AT, cross
healthy_peak = max(p for tt, p in zip(ts, phis) if 5.0 < tt < DEAD_AT)
assert healthy_peak < 8, f'phi reached {healthy_peak:.1f} on a healthy node'
assert phis[-1] > healthy_peak
print(f'peak phi while alive: {healthy_peak:.2f}; detection delay {cross - DEAD_AT:.1f}s')

## 2. The killer feature: surviving a network blip

Now we run the **same node**, still alive, but the network drops every heartbeat for 1.5 seconds
in the middle (`blip = (15s, 1.5s)`). A fixed `timeout = 1.0s` (the most natural "1 missed beat"
rule) screams FAILURE and triggers an unnecessary failover — a classic cause of split-brain.

Watch what phi does: it *climbs* during the blip — appropriate suspicion — and then snaps back
to ~0 as soon as heartbeats resume. **Whether it crossed your threshold on the way up is a
tuning question, not a property of phi**, and the code below prints the answer rather than
assuming it. (The dedicated `phi-accrual-failure-detection` lab pushes on this: on a link with
this much jitter, the Cassandra default of φ=8 *is* crossed by a 1.5s blip.)

In [ ]:
random.seed(0)
blip_trace = make_trace(blip=(15.0, 1.5))

# --- fixed-timeout detector for comparison ---
def fixed_timeout_states(heartbeats, timeout, total=TOTAL, sample_every=0.1):
    last_seen = None
    hb_iter = iter(heartbeats)
    next_hb = next(hb_iter, None)
    t = 0.0
    out_t, out_dead = [], []
    while t < total:
        while next_hb is not None and next_hb <= t:
            last_seen = next_hb
            next_hb = next(hb_iter, None)
        is_dead = last_seen is not None and (t - last_seen) > timeout
        out_t.append(t); out_dead.append(1 if is_dead else 0)
        t += sample_every
    return out_t, out_dead

ft_t, ft_dead = fixed_timeout_states(blip_trace, timeout=1.0)

# --- phi detector on the same trace ---
detector = PhiDetector()
ts2, phis2 = [], []
hb_iter = iter(blip_trace)
next_hb = next(hb_iter, None)
t = 0.0
while t < TOTAL:
    while next_hb is not None and next_hb <= t:
        detector.heartbeat(next_hb)
        next_hb = next(hb_iter, None)
    ts2.append(t); phis2.append(detector.phi(t))
    t += 0.1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
ax1.step(ft_t, ft_dead, color='tab:red')
ax1.set_ylabel('fixed timeout=1.0s\n(0=alive, 1=DEAD)')
ax1.set_title('Fixed timeout vs phi accrual under a 1.5s network blip (node still alive!)')
ax1.axvspan(15.0, 16.5, color='grey', alpha=0.2, label='network blip')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

ax2.plot(ts2, phis2, label='phi')
ax2.axhline(8, color='red', linestyle='--', label='threshold (phi=8)')
ax2.axvspan(15.0, 16.5, color='grey', alpha=0.2)
ax2.set_xlabel('time (s)'); ax2.set_ylabel('phi')
ax2.legend(loc='upper right'); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

fp_fixed = sum(1 for d in ft_dead if d == 1)
max_phi_during_blip = max(p for tt, p in zip(ts2, phis2) if 15.0 <= tt < 17.0)
after_blip = max(p for tt, p in zip(ts2, phis2) if 18.0 <= tt < 25.0)
print(f'fixed-timeout detector spent {fp_fixed*0.1:.1f}s falsely declaring DEAD during the blip')
print(f'phi accrual peak during blip: {max_phi_during_blip:.2f}')
print(f'phi once heartbeats resume  : {after_blip:.2f}')

assert fp_fixed * 0.1 > 1.0, 'the 1.0s timeout should have false-alarmed for over a second'
assert max_phi_during_blip > 1.0, 'phi should get suspicious during a real outage'
assert after_blip < 1.0, 'phi should collapse once beats resume'

# Whether this counts as a "false alarm" depends entirely on the threshold, and
# THAT is the honest headline. Report it rather than assuming an answer.
verdict = 'WOULD false-alarm' if max_phi_during_blip > 8 else 'would NOT false-alarm'
print(f'\nat threshold 8 phi {verdict} here (peak {max_phi_during_blip:.2f})')
print('Phi does not abolish false positives — it re-denominates the knob from')
print('"seconds of silence" into "how improbable", which transfers between links')
print('and lets different callers pick different confidence levels from one signal.')
print('The phi-accrual-failure-detection lab digs into exactly where that breaks down.')

## 3. Tuning the threshold

The threshold is the only knob. Sensible defaults you'll see in the wild:

| Threshold | Meaning | Used by |
|---|---|---|
| **φ > 1** | ~10% — *very* sensitive, lots of suspicions | aggressive monitoring |
| **φ > 8** | ~10⁻⁸ — Cassandra default | Cassandra, ScyllaDB |
| **φ > 12** | ~10⁻¹² — Akka default for cluster membership | Akka |

Higher threshold ⇒ fewer false positives, slightly slower detection. Same trade-off as notebook
1 — phi does not delete the Pareto curve — but the *units* are now probability instead of
seconds, which is what makes one setting travel between a LAN and a WAN.

The second knob is `min_std`. Floor it too low for a lossy link and an ordinary packet-loss
burst reads as a many-sigma event; too high and you go slow to detect real deaths.

## ✅ Recap

- **Fixed timeout**: simple, but you must hand-tune for the worst-case network. Same threshold across a fast LAN and a flaky WAN won't work.
- **Phi accrual**: *learns* the normal cadence, so a node that is merely slower is no longer
  mistaken for a dead one, and the threshold means roughly the same thing across environments.
  It does **not** eliminate false positives, and it has two knobs (threshold and `min_std`),
  not zero.
- The mental model: instead of *"node X is dead"* we say *"we are increasingly suspicious of node X, and at some confidence we will act on that suspicion."*

👉 Next: notebook 3 zooms out from "how to decide" to **"how do real heartbeats get wired up?"** — push vs pull, central monitor vs gossip, and the surprisingly subtle issue of *what to do once you suspect a node is dead*.